In [21]:
import numpy as np 
import pandas as pd 
import random
import joblib 


In [22]:
states = [
    "start", 
    "budget_known", 
    "usage_known",
    "specs_known", 
    "ready_to_recommend"
]

state_to_index = {s: i for i, s in enumerate(states)}

In [23]:

actions = [ 
    "ask_budget", 
    "ask_usage",
    "ask_specs", 
    "recommend"
]

action_to_index = {a: i for i, a in enumerate(actions)}

In [24]:
def get_reward(state, action):
    
    if action == "recommend" and state == "ready_to_recommend":
        return 10
    elif action == "recommend":
        return -5
    elif action.startswith("ask"):
        return -1
    return 0

In [25]:
def get_next_state(state, action):
    
    if state == "start" and action == "ask_budget":
        return "budget_known"
    elif state == "budget_known" and action == "ask_usage":
        return "usage_known"
    elif state == "usage_known" and action == "ask_specs":
        return "specs_known"
    elif state == "specs_known" and action == "recommend":
        return "ready_to_recommend"
    return state

In [26]:
Q = np.zeros((len(states), len(actions)))

alpha = 0.1
gamma = 0.9
epsilon = 0.2 

In [27]:
episodes = 500

for _ in range(episodes):
    state = "start"

    for _ in range(10):
        s = state_to_index[state]

        if random.random() < epsilon:
            action = random.choice(actions)
        else:
            action = actions[np.argmax(Q[s])]

        a = action_to_index[action]

        next_state = get_next_state(state, action)
        ns = state_to_index[next_state]

        reward = get_reward(state, action)

        Q[s, a] = Q[s, a] + alpha * (
            reward + gamma * np.max(Q[ns]) - Q[s, a]
        )

        state = next_state

        if state == "ready_to_recommend":
            break

In [28]:
for s in states:
    s_idx = state_to_index[s]
    best_action = actions[np.argmax(Q[s_idx])]
    print(f"{s} → {best_action}")

start → ask_budget
budget_known → ask_usage
usage_known → ask_specs
specs_known → recommend
ready_to_recommend → ask_budget


In [29]:
joblib.dump(Q, "q_table.pkl")
joblib.dump(states, "states.pkl")
joblib.dump(actions, "actions.pkl")

print("Q-table and policy saved successfully.")

Q-table and policy saved successfully.
